# T1 - Deep Learning 

Feito por: Louise e Luca


Numpy faz as oporacoes de matrizes para nos, entao nos temos que criar uma classe base (pytorch chama de tensor) 

tensor --> matriz pessos + matriz de gradiente 

In [58]:
import numpy as np

Unidade basica - matriz que guarda as informcoes para a fazer o back propagation depois 

In [59]:
class Tensor:
    def __init__(self, dimencao):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.random.random(dimencao)

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        return self.peso + outra.peso

In [60]:
dimencao = [2,3]
exemplo = Tensor(dimencao)

print(exemplo)
print(exemplo.gradiente)

[[0.29031636 0.71846791 0.90823626]
 [0.7519356  0.73970776 0.92985787]]
[[0.44482562 0.06077217 0.51356768]
 [0.06967223 0.07762753 0.95002097]]


In [61]:
exemplo2 = Tensor(dimencao)

In [62]:
exemplo + exemplo2

array([[0.33299268, 0.93918354, 1.52414145],
       [1.32764073, 0.82087636, 1.06910107]])

Agora vamos mexer no gradiente 
queremos impemetar as funcoes basicas para qaundo fizermos o back propagetion qualquer funcao ser diferenciavel (derivar )

uma forma de ccalular o gradiente: graudar as regras da derivada 


In [63]:
class Tensor:
    def __init__(self, dimencao):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.random.random(dimencao)
        self.regra = 0

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        self.regra = 1 
        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso + outra.peso

        return retorno

    def __mul__(self, outra):  # ponto a ponto 

        self.regra = outra.peso
        outra.regra = self.peso


        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso * outra.peso
 
        return retorno
    

    

In [64]:
dimencao = [2,2]
teste1 = np.array([[1,2],[3,4]])
teste2 = np.array([[1,1],[1,1]])

exemplo_3 = Tensor(dimencao)
exemplo_4 = Tensor(dimencao)

exemplo_3.peso = teste1
exemplo_4.peso = teste2

print(exemplo_3, "\n\n", exemplo_4)

[[1 2]
 [3 4]] 

 [[1 1]
 [1 1]]


In [65]:
exemplo_4 * exemplo_3

In [66]:
exemplo_3.regra

array([[1, 1],
       [1, 1]])

In [67]:
exemplo_4.regra

array([[1, 2],
       [3, 4]])

Agora temos o nosso gradiente local, nos chamamos isso de regra, aqui esta uma teste para ver se esta funcionando:

dervada em relacao a 3 da funcao: ( (3 * 4) * 5 ) = 20 

In [68]:
dimencao = [1]
num3 = np.array([3])
num4 = np.array([4])
num5 = np.array([5])

exemplo_num3 = Tensor(dimencao)
exemplo_num4 = Tensor(dimencao)
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

print(exemplo_num3, "\n", exemplo_num4, "\n", exemplo_num5)
print( exemplo_num3 * exemplo_num4 * exemplo_num5)

print(exemplo_num5.regra) # 3 * 4 = 12 o gradiente esta certo
exemplo_num5.gradiente = exemplo_num5.regra 

print(exemplo_num5.gradiente)

print(exemplo_num3.regra)
exemplo_num3.gradiente = exemplo_num3.regra * exemplo_num5.peso

print(exemplo_num3.gradiente)

[3] 
 [4] 
 [5]
[60]
[12]
[12]
[4]
[20]


In [69]:
exemplo_3.peso.shape

(2, 2)

Agora nos queremos que atualize o gradiente automaticamente, sem ter que fazer 
```python 
exemplo_num3.gradiente = exemplo_num3.regra * exemplo_num5.peso
```

para isso a gente vai mudar para fazwe uma funcao= gradiente local * outro

In [70]:
class Tensor:
    def __init__(self, dimencao):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda x:0
        self.prox = None

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso + outra.peso
        self.prox = retorno # agora temos um grafo

        def regra(tensor):
            tensor.gradiente += self.prox.gradiente

        self.regra = lambda: regra(self)
        outra.regra = lambda: regra(outra)

        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso * outra.peso
        self.prox = retorno

        def regra():
            self.gradiente += self.prox.gradiente * outra.peso # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += outra.prox.gradiente * self.peso 
 
        return retorno

    def backward(self): # calulo dos gradientes
        pass



    

In [71]:
dimencao = [1]
num3 = np.array([3])
num4 = np.array([4])
num5 = np.array([5])

exemplo_num3 = Tensor(dimencao)
exemplo_num4 = Tensor(dimencao)
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

teste = exemplo_num3 + exemplo_num4
teste.gradiente = np.ones_like(teste, float) 
exemplo_num3.regra()
print(exemplo_num3.gradiente)

[1.]


Vamos trabalhar numa forma coesa de fazer o backward pass automaticmaente, depois voltamos a implementar funções. Usaremos o algoritmo de ordenação topologica baseado em DFS. Para isso vamos mudar a lógica de prox, e definir um anterior, pois é consideravelmente mais facil trabalhar na ordem inversa

In [84]:
class Tensor:
    def __init__(self, dimencao, filhos= ()):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda: None
        self.anteriores = set(filhos)  # Adicionamos isso para poder calcular a ordem topologica
        self.prox = None

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        retorno = Tensor(self.peso.shape, (self, outra))
        retorno.peso = self.peso + outra.peso
        self.prox = retorno # agora temos um grafo

        def regra():
            self.gradiente += retorno.gradiente
            outra.gradiente += retorno.gradiente
            
        retorno.regra = regra
        
        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        retorno = Tensor(self.peso.shape, (self, outra))
        retorno.peso = self.peso * outra.peso
        self.prox = retorno

        def regra():
            self.gradiente += retorno.gradiente * outra.peso # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += retorno.gradiente * self.peso 
        
        retorno.regra = regra
 
        return retorno

    def backward(self): # calulo dos gradientes
        self.gradiente = np.array(self.peso.shape)
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.anteriores:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        for nodo in reversed(topo):
            nodo.regra()


In [85]:
dimencao = [1]
num3 = np.array([3])
num4 = np.array([4])
num5 = np.array([5])

exemplo_num3 = Tensor(dimencao)
exemplo_num4 = Tensor(dimencao)
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

teste = exemplo_num3 + exemplo_num4
final = exemplo_num5 * teste
final.backward()
print(exemplo_num3.gradiente)

[5.]
